# <center> **Data Factory** 🏭

This notebook is dedicated to generating synthetic data for training the **intents model** used by the AI assistant. 
    
**Synthetic data** is helpful when there is not enough real-world data available or when there is a need to expand the dataset for better model generalization.

---
    
### Why Synthetic Data?
Synthetic data generation is often used in cases where:
- Limited labeled data is available.
- Data privacy is a concern.
- Bias in the real-world data needs to be balanced by adding more diverse samples.

For generating data for the AI assistant, it can help simulate user interactions and intents in a way that mirrors the potential inputs of real customers, while ensuring the dataset remains scalable and diverse.

> Note: Before starting synthetic data generation, it’s critical to ensure that the generated data mimics the patterns of real user queries as much as possible. Misaligned synthetic data can negatively impact model performance.

In [4]:
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import requests
import random
import time
import json
import sys
import os

ModuleNotFoundError: No module named 'tqdm'

### **Configuration**

In [2]:
NAME = "conversational"
LANGUAGE = "esp"

DIR = ""
CONFIG_PATH = os.path.join(DIR, "config/")

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
API_URL = "https://llama-llm-ap1001-llama-llm.apps.cluster-pfgjj.pfgjj.sandbox1803.opentlc.com/v1/chat/completions"
API_KEY = "test"

### **Read Files**

In [3]:
def read_md(file_path):
    # Read the content of a Markdown file
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

def chunk_text(text_list):
    chunks = []
    current_chunk = []
    for text in text_list:
        current_chunk.append(text)
        if text == "": 
            chunks.append(''.join(current_chunk))
            current_chunk = []
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    chunks = [chunk for chunk in chunks if chunk]
    return chunks

In [4]:
# read JSON file (input format)
file_path = os.path.join(CONFIG_PATH, NAME, f'intents/config.json')
with open(file_path, 'r') as file:
    config = json.load(file)

# read TXT file (random words)
file_path = os.path.join(CONFIG_PATH, 'random/', f'random-words-{LANGUAGE}.txt')
with open(file_path, 'r') as file:
    random_words = list(file)
    random_words = [word.strip("\n") for word in random_words]

### **Set up inference API**

In [5]:
def call_LLM(system_prompt, user_prompt, temperature, max_tokens=150):

    headers = {
        #"Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": messages,
        "max_tokens": max_tokens
    }
    
    # Send the POST request to the API
    response = requests.post(API_URL, headers=headers, json=payload)
    
    # Check for errors
    if response.status_code == 200:
        result = response.json()
        return result['choices'][0]['message']['content'].strip()
    else:
        print(f"Error: {response.status_code}")
        print(response.text)

### **Test prompts**

### **Synthetic data generator**

In [ ]:
%%time

# fix parameter values
iterations = config["iterations"]
max_temp = config["max-temperature"]
max_rep = config["max-repetition"]

intent_dict = config["intents"] # {conversational:..., irrelevant;..., spam:...}
intent_keys = list(intent_dict.keys()) # ["coneversational", "irrelevant", "spam"]

# define dict to store generated data
synthetic_data = {
    "text": [],
    "label": []
}

# main loop
for iteration in tqdm(range(iterations)):
    
    # randomly define text variables for this iteration
    rep = random.choice(range(2,max_rep+1))
    temp = round(random.uniform(0, max_temp),2)
    
    intent = random.choice(intent_keys)
    word = random.choice(random_words)
    
    mood = random.choice(config["moods"])
    personality = random.choice(config["personalities"])
    
    
    # format prompt templates
    if intent=="conversational":
        conversation_topic = "general" #random.choice(list(intent_dict["conversational"].keys()))
        
        context = random.choice(intent_dict["conversational"][conversation_topic]["context"])
        area = random.choice(intent_dict["conversational"][conversation_topic]["area"])
        length = random.choice(intent_dict["conversational"][conversation_topic]["length"])
        action = random.choice(intent_dict["conversational"][conversation_topic]["action"])
        
        system_prompt = intent_dict["conversational"][conversation_topic]["system"].format(personality=personality, mood=mood)
        user_prompt = intent_dict["conversational"][conversation_topic]["user"].format(context=context, area=area, length=length, action=action)

    elif intent=="irrelevant":
        context = random.choice(intent_dict["irrelevant"]["context"])
        system_prompt = intent_dict["irrelevant"]["system"].format(personality=personality, mood=mood)
        user_prompt = intent_dict["irrelevant"]["user"].format(context=context).format(word=word)
    
    elif intent=="spam":
        context = random.choice(intent_dict["spam"]["context"])
        system_prompt = intent_dict["spam"]["system"].format()
        try: 
            user_prompt = intent_dict["spam"]["user"].format(context=context).format(word=word, repetition=rep)
        except IndexError:
            user_prompt = intent_dict["spam"]["user"].format(context=context)
        
    # generate text
    text = call_LLM(system_prompt=system_prompt,
                    user_prompt=user_prompt,
                    temperature=temp
                   )
    
    # store text and label in dict
    synthetic_data["text"] += [text]
    synthetic_data["label"] += [intent]
        
print("_"*50)

 59%|█████▉    | 588/1000 [26:23<22:13,  3.24s/it]  

### **Export files**

In [ ]:
# convert dict into JSON with label studio valid format

output = []

for i in range(iterations):
    # read JSON file (format dictionary)
    file_path = os.path.join(CONFIG_PATH, NAME, 'intents/format.json')
    with open(file_path, 'r') as file:
        format_ = json.load(file)

    # plug values in dict
    format_["id"] += i
    format_["data"]["text"] = synthetic_data["text"][i]
    format_["annotations"][0]["result"][0]["value"]["choices"][0] = synthetic_data["label"][i]

    # append dict to list
    output += [format_.copy()]

# get the current time
current_datetime = datetime.now().strftime("[%y-%m-%d %H:%M:%S]")

# write JSON file
file_path = os.path.join(DIR, f'generated/{NAME} {current_datetime}.json')
with open(file_path, 'w') as file:
    json.dump(output, file)
    
print("Synthetic dataset generated OK.")

In [ ]:
# also export to CSV :)
import pandas as pd

file_path = os.path.join(DIR, f'generated/{NAME} {current_datetime}.csv')
df = pd.DataFrame(synthetic_data)
df.to_csv(file_path, index=False)

---